In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import(
    col,
    row_number,
    trim
)
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

#### Importing validation class from the test_utils module

In [0]:
from validation_utils.test_utils import Validations
obj = Validations()

#### Defining orchestrator function that runs all the test functions inside the test_utils

In [0]:
def orchestrate_tests(
    table: str,
    obj: Validations,
    primary_col: str,
    dedup_cols: list,
    cdc: str
) -> None:
    df = spark.table(f"lakehouse.silver.{table}")

    def log(msg: str) -> None:
        print(f"\n{'-' * 20}\t{msg}\t{'-' * 20}\n")

    log("Null Check Start")
    obj.null_check(df, primary_col)
    log("Null Check End")

    log("Duplicate Check Start")
    obj.duplicate_check(df, dedup_cols, cdc)
    log("Duplicate Check End")

    log("Whitespace Check Start")
    obj.whitespace_check(df)
    log("Whitespace Check End")



#### 1. erp_customers

In [0]:
orchestrate_tests(
    table = "erp_customers",
    obj = obj,
    primary_col = 'customer_number',
    dedup_cols = ['customer_number'],
    cdc = 'birth_date'
)

##### Checking the missing records in other tables 

In [0]:
if spark.sql("""
             SELECT 1
             FROM lakehouse.silver.erp_customers
             WHERE birth_date > current_date()
             """).count() > 0:
    raise Exception("Birth date is greater than current date")
else:
    print("erp_customers.birth_date is valid")

#### 2. erp_customer_location

In [0]:
orchestrate_tests(
    table = "erp_customer_location",
    obj = obj,
    primary_col = 'customer_number',
    dedup_cols = ['customer_number'],
    cdc = 'ingestion_ts'
)

##### Checking the missing records in other tables

In [0]:
if spark.sql("""
             SELECT 1
             FROM lakehouse.silver.erp_customer_location
             WHERE customer_number NOT IN(
                 SELECT customer_number
                 FROM lakehouse.silver.crm_customers
             )
             """).count() > 0:
    raise Exception("customer_number in erp_customer_location is not in crm_customers")
else:
    print("customer_number in erp_customer_location is valid")

#### 3. erp_product_category

In [0]:
orchestrate_tests(
    table = "erp_product_category",
    obj = obj,
    primary_col = 'category_id',
    dedup_cols = ['category_id'],
    cdc = 'ingestion_ts'
)